In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [6]:
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|             grupo|                item|pagina|valor|
+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|69dd821edcd773d51...|  Romance|2026-04-13 23:53:59|Mercado Laboral TI|     The Requiem Red|     1|22.65|
|69dd821edcd773d51...|  Romance|2026-04-13 23:53:59|Mercado Laboral TI|         Set Me Free|     1|17.46|
|69dd821edcd773d51...|  Romance|2026-04-13 23:53:59|Mercado Laboral TI|The Natural Histo...|     1|45.22|
|69dd821edcd773d51...|  Romance|2026-04-13 23:53:59|Mercado Laboral TI|   Obsidian (Lux 

In [7]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 56
Registros validos: 56


In [8]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|     The Requiem Red|22.65|
|         Set Me Free|17.46|
|The Natural Histo...|45.22|
|   Obsidian (Lux #1)|14.86|
|             Burning|28.81|
|A Fierce and Subt...|28.13|
|Scarlett Epstein ...|43.55|
|   Nightingale, Sing|38.28|
|Library of Souls ...|48.56|
|Frostbite (Vampir...|29.99|
|          Wild Swans|14.36|
|Until Friday Nigh...|46.31|
|This Is Where It ...|27.12|
|     The Darkest Lie|35.35|
|    My Kind of Crazy|40.36|
|    Don't Get Caught|55.35|
|Catching Jordan (...|50.83|
|Aristotle and Dan...|58.14|
|The Epidemic (The...|14.44|
|Stars Above (The ...|48.05|
+--------------------+-----+
only showing top 20 rows



In [9]:
df.filter(df["valor"] > 50).show()

+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|             grupo|                item|pagina|valor|
+--------------------+---------+-------------------+------------------+--------------------+------+-----+
|69dd821edcd773d51...|  Romance|2026-04-13 23:54:00|Mercado Laboral TI|    Don't Get Caught|     1|55.35|
|69dd821edcd773d51...|  Romance|2026-04-13 23:54:01|Mercado Laboral TI|Catching Jordan (...|     1|50.83|
|69dd821edcd773d51...|  Romance|2026-04-13 23:54:01|Mercado Laboral TI|Aristotle and Dan...|     1|58.14|
|69dd821edcd773d51...|  Romance|2026-04-13 23:54:02|Mercado Laboral TI|No Love Allowed (...|     2|54.65|
|69dd821edcd773d51...|  Romance|2026-04-13 23:54:02|Mercado Laboral TI|Exit, Pursued by ...|     2|51.34|
|69dd821edcd773d51...|  Romance|2026-04-13 23:54:02|Mercado Laboral TI|      The Alien Club|     2| 54.4|
|69dd821edcd773d51...|  Romance|2026-04-13 23:

In [10]:
df.groupBy("grupo").count().show()

+------------------+-----+
|             grupo|count|
+------------------+-----+
|Mercado Laboral TI|   56|
+------------------+-----+



In [11]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+-----------------+-------------+-------------+
|categoria|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+-----------------+-------------+-------------+
|  Romance|             56|35.87232142857143|         10.0|        58.14|
+---------+---------------+-----------------+-------------+-------------+



In [12]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "categoria", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: Aristotle and Dante Discover the Secrets of the Universe (Aristotle and Dante Discover the Secrets of the Universe #1)
Categoría: Romance
Fecha y hora de captura: 2026-04-13 23:54:01
Valor: 58.14


In [16]:
from pyspark.sql import functions as F

NOMBRE_GRUPO= "Mercado Laboral TI"
# CONSULTA DE SALIDA: Resumen de integridad por Categoría
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: Mercado Laboral TI ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|  Romance|          56|       35.87|  2026-04-13 23:59:57|
+---------+------------+------------+---------------------+

